# Pixels to Predictions: CS-GY 6953 (Deep Learning) Kaggle Competition
## MODEL 6
### Author: Mariia Onokhina

---
### Experiment Results
- **Model:** `HuggingFaceTB/SmolVLM-500M-Instruct`
- **Prompt:** `lecture_100` prompt with image-careful instruction, metadata, hint, and first 100 words of `lecture`
- **Training:** Contrastive answer-index ranking fine-tuning with `LR = 5e-6`, `GRAD_ACCUM_STEPS = 8`
- **LoRA:** `attn_r8`, `r=8`, `lora_alpha=16`, `lora_dropout=0.05`, `target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]`
- **Trainable params:** Under 5M
- **Objective:** For each row, score appended answer indices `" 0"`, `" 1"`, etc.; treat scores as logits; train with cross-entropy against the correct answer index
- **Epochs trained:** 2
- **Best validation accuracy:** `0.6708015267` at epoch 2
- **Public Kaggle score:** `0.70422`
- **Decision:** Best model so far; strong improvement over Model 5 (`0.64587` public), use as the new baseline for Model 7/ensembles

---
**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

## Installation of Libraries 

Before installing any libraries, I create a new Conda environment and add it to Jupyter Notebook to ensure I start from a clean slate and that the code is reproducible.

**Run the following code in a terminal if you'd like to start fresh with a new environment:**
```bash
conda create -n pixels-to-predictions python=3.10
conda activate pixels-to-predictions
conda install -c conda-forge notebook
conda install -c conda-forge ipykernel
python -m ipykernel install --user --name pixels-to-predictions --display-name "Pixels-to-predictions DL"
```

IMPORTANT: Manually change the Kernel in Jupyter Notebook in VS Code or Jupyter Lab to "Pixels-to-predictions DL".

In [1]:
# Uncomment this cell to install the necessary Python packages.
import sys
print("Python:", sys.executable)
!{sys.executable} -m pip install -q transformers==4.57.6 peft==0.18.1 kaggle matplotlib scikit-learn pandas numpy ipywidgets jupyterlab_widgets bitsandbytes accelerate datasets pillow --quiet

Python: /home/devvingduo/miniforge3/envs/pixels-to-predictions/bin/python


---
## Imports & Configuration

In [18]:
# Imports & Configuration
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import ast
from transformers import AutoProcessor, AutoModelForVision2Seq

# For LoRa fine-tuning
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm   # For progress bar
from itertools import combinations
import gc

In [3]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [4]:
# This code makes sure it uses only 1 GPU of my choice
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
# Paths 
DATA_DIR = Path("../data")
IMAGE_ROOT = DATA_DIR / "images"

# Model
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Basic Settings
IMG_SIZE = 224

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


---
### Load and Preprocess Data

Download data from https://www.kaggle.com/competitions/pixels-to-predictions/data via Kaggle CLI.

For this, you need a Legacy API key which you can create here: https://www.kaggle.com/settings.

When you create a new key, it will download a ```kaggle.json```. 

In your terminal, run:
```bash
mkdir -p ~/.kaggle
mv <your_downloads_folder> ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json
```

Verify that it worked by running: 
```bash
ls -la ~/.kaggle
```

I have to add it manually because I'm working via SSH into my Linux server machine.

In [7]:
# Uncomment this cell to download the data in a .zip file
#!kaggle competitions download -c pixels-to-predictions

In [8]:
# Uncomment this cell to unzip the data into "data" folder
#!unzip pixels-to-predictions.zip -d data

In [9]:
# Load CSVs
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

Inspect the data. 

In [10]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3109 entries, 0 to 3108
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           3109 non-null   object
 1   image_path   3109 non-null   object
 2   question     3109 non-null   object
 3   choices      3109 non-null   object
 4   num_choices  3109 non-null   int64 
 5   answer       3109 non-null   int64 
 6   hint         2385 non-null   object
 7   lecture      2669 non-null   object
 8   solution     2580 non-null   object
 9   task         3109 non-null   object
 10  grade        3109 non-null   object
 11  subject      3109 non-null   object
 12  topic        3109 non-null   object
 13  category     3109 non-null   object
 14  skill        3109 non-null   object
dtypes: int64(2), object(13)
memory usage: 364.5+ KB


In [11]:
# Function to parse the choices column using ast module (Abstract Syntax Tree)
def parse_choices(x):
    # If x is already a list, return it
    if isinstance(x, list):
        return x
    # If x is a string, parse it using ast.literal_eval
    return ast.literal_eval(x)

The ```choices``` column is a JSON string, so we parse it using the function above.

In [12]:
for df in [train_df, val_df, test_df]:
    df["choices_list"] = df["choices"].apply(parse_choices)

---
### Prompt

Keep `build_prompt_model2_imagecareful` which is a variation of Model 2's prompt. 

We want to see whether including lecture field in the prompt improves the score. It's very long so we truncate it.

In [13]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in ["nan", "none", "null"]:
        return ""
    return x

In [14]:
def truncate_words(x, max_words=180):
    x = clean_text(x)
    if not x:
        return ""
    words = x.split()
    return " ".join(words[:max_words])

In [16]:
# Try "no_lecture", "lecture_100", "lecture_180", "lecture_250"
PROMPT_STYLE = "lecture_180"


def build_prompt(row):
    choices = row["choices_list"]

    choice_lines = "\n".join(
        [f"{i}: {choice}" for i, choice in enumerate(choices)]
    )

    prompt = "<image>\n"
    prompt += "Answer the science multiple-choice question.\n"
    prompt += "Look carefully at the image when it is relevant.\n"
    prompt += "Use the provided lecture and hint only if helpful.\n"
    prompt += "Return only the number of the correct choice.\n\n"

    grade = clean_text(row.get("grade", ""))
    subject = clean_text(row.get("subject", ""))
    topic = clean_text(row.get("topic", ""))
    category = clean_text(row.get("category", ""))
    skill = clean_text(row.get("skill", ""))

    if grade:
        prompt += f"Grade: {grade}\n"
    if subject:
        prompt += f"Subject: {subject}\n"
    if topic:
        prompt += f"Topic: {topic}\n"
    if category:
        prompt += f"Category: {category}\n"
    if skill:
        prompt += f"Skill: {skill}\n"

    lecture = clean_text(row.get("lecture", ""))

    if PROMPT_STYLE == "no_lecture":
        lecture = ""
    elif PROMPT_STYLE == "lecture_100":
        lecture = truncate_words(lecture, max_words=100)
    elif PROMPT_STYLE == "lecture_180":
        lecture = truncate_words(lecture, max_words=180)
    elif PROMPT_STYLE == "lecture_250":
        lecture = truncate_words(lecture, max_words=250)
    else:
        raise ValueError(f"Unknown PROMPT_STYLE: {PROMPT_STYLE}")

    if lecture:
        prompt += f"\nLecture:\n{lecture}\n"

    hint = clean_text(row.get("hint", ""))
    if hint:
        prompt += f"\nHint:\n{hint}\n"

    prompt += f"\nQuestion:\n{clean_text(row['question'])}\n\n"
    prompt += f"Choices:\n{choice_lines}\n\n"
    prompt += "Answer:"

    return prompt

---
### Model Loading

We will try several versions of LoRA with different hyperparameters to choose the best one.

In [17]:
MODEL6_LORA_NAME = "attn_r8"

LORA_CONFIGS = {
    "qv_r8": {
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "target_modules": ["q_proj", "v_proj"],
        "lr": 1e-5,
    },
    "attn_r8": {
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "lr": 5e-6,
    },
    "attn_mlp_r4": {
        "r": 4,
        "lora_alpha": 8,
        "lora_dropout": 0.05,
        "target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        "lr": 5e-6,
    },
}

cfg = LORA_CONFIGS[MODEL6_LORA_NAME]

print("Model 6 LoRA config:")
print(cfg)

Model 6 LoRA config:
{'r': 8, 'lora_alpha': 16, 'lora_dropout': 0.05, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'], 'lr': 5e-06}


Load Model 6 model from scratch.

In [19]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

model.config.use_cache = False

for p in model.parameters():
    p.requires_grad = False

lora_config = LoraConfig(
    r=cfg["r"],
    lora_alpha=cfg["lora_alpha"],
    lora_dropout=cfg["lora_dropout"],
    bias="none",
    target_modules=cfg["target_modules"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

assert trainable < 5_000_000

/home/devvingduo/miniforge3/envs/pixels-to-predictions/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Trainable params: 2,080,768
Total params: 509,563,072
Trainable %: 0.4083%


---
### Index-Only Scoring

In [20]:
@torch.no_grad()
def score_indices_for_row(row, prompt_builder, score_mode="sum"):
    image = Image.open(IMAGE_ROOT / row["image_path"]).convert("RGB")
    prompt = prompt_builder(row)

    num_choices = int(row["num_choices"])
    answer_texts = [f" {i}" for i in range(num_choices)]
    full_texts = [prompt + ans for ans in answer_texts]

    prompt_inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
        padding=True,
    )

    prompt_len = prompt_inputs["input_ids"].shape[1]

    full_inputs = processor(
        text=full_texts,
        images=[image] * num_choices,
        return_tensors="pt",
        padding=True,
    )

    full_inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in full_inputs.items()
    }

    input_ids = full_inputs["input_ids"]
    attention_mask = full_inputs.get("attention_mask", torch.ones_like(input_ids))

    outputs = model(**full_inputs)
    logits = outputs.logits.float()

    shift_logits = logits[:, :-1, :]
    shift_input_ids = input_ids[:, 1:]
    shift_attention = attention_mask[:, 1:]

    log_probs = F.log_softmax(shift_logits, dim=-1)

    token_log_probs = log_probs.gather(
        dim=-1,
        index=shift_input_ids.unsqueeze(-1),
    ).squeeze(-1)

    answer_mask = torch.zeros_like(shift_input_ids, dtype=torch.bool)

    start = max(prompt_len - 1, 0)
    answer_mask[:, start:] = True

    answer_mask = answer_mask & shift_attention.bool()

    scores = []

    for i in range(num_choices):
        vals = token_log_probs[i][answer_mask[i]]

        if vals.numel() == 0:
            scores.append(-1e9)
        elif score_mode == "sum":
            scores.append(vals.sum().item())
        elif score_mode == "mean":
            scores.append(vals.mean().item())
        else:
            raise ValueError("score_mode must be 'sum' or 'mean'")

    return scores

In [21]:
# Since we will train the model this time, we need to use the training version of the scoring function.
# Same idea as Model 4 scoring, but WITHOUT torch.no_grad(), so LoRA can learn from the choice scores.
def score_indices_for_row_train(row, prompt_builder, score_mode="sum"):
    prompt = prompt_builder(row)

    num_choices = int(row["num_choices"])
    image_path = IMAGE_ROOT / row["image_path"]
    image = Image.open(image_path).convert("RGB")

    scores = []

    for choice_idx in range(num_choices):
        answer_text = " " + str(choice_idx)
        full_text = prompt + answer_text

        prompt_inputs = processor(
            text=[prompt],
            images=[image],
            return_tensors="pt",
        )

        prompt_len = prompt_inputs["input_ids"].shape[1]

        inputs = processor(
            text=[full_text],
            images=[image],
            return_tensors="pt",
        )

        model_device = next(model.parameters()).device

        inputs = {
            k: v.to(model_device) if torch.is_tensor(v) else v
            for k, v in inputs.items()
        }

        input_ids = inputs["input_ids"]

        outputs = model(**inputs)
        logits = outputs.logits.float()

        log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
        target_ids = input_ids[:, 1:]

        token_log_probs = log_probs.gather(
            dim=2,
            index=target_ids.unsqueeze(-1),
        ).squeeze(-1)

        start = max(prompt_len - 1, 0)
        answer_token_log_probs = token_log_probs[:, start:]

        if score_mode == "sum":
            score = answer_token_log_probs.sum()
        elif score_mode == "mean":
            score = answer_token_log_probs.mean()
        else:
            raise ValueError("score_mode must be 'sum' or 'mean'")

        scores.append(score)

    return torch.stack(scores)

In [22]:
def predict_row(row, prompt_builder, score_mode="sum"):
    scores = score_indices_for_row(row, prompt_builder, score_mode=score_mode)
    pred = int(np.argmax(scores))
    return pred, scores

---
### Evaluation

Contrastive LoRA training scores every answer index for the same question, treats those scores as classification logits, and uses cross-entropy to push the correct answer’s score above the wrong choices. 

In [23]:
# Score all answer indices, then apply cross entropy to make the correct answer index score highest
def contrastive_loss_for_row(row, prompt_builder=build_prompt):
    scores = score_indices_for_row_train(
        row,
        prompt_builder=prompt_builder,
        score_mode="sum",
    )

    logits = scores.unsqueeze(0)

    target = torch.tensor(
        [int(row["answer"])],
        dtype=torch.long,
        device=logits.device,
    )

    loss = F.cross_entropy(logits, target)

    pred = int(torch.argmax(scores.detach()).item())

    return loss, pred, scores.detach().cpu().numpy()

In [24]:
def evaluate_model(df=None, max_rows=None, score_mode="sum"):
    if df is None:
        df = val_df

    if max_rows is not None:
        eval_df = df.sample(max_rows, random_state=SEED).reset_index(drop=True)
    else:
        eval_df = df.reset_index(drop=True)

    model.eval()

    preds = []
    labels = []
    all_scores = []

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="evaluate"):
        pred, scores = predict_row(
            row,
            build_prompt,
            score_mode=score_mode,
        )

        preds.append(pred)
        labels.append(int(row["answer"]))
        all_scores.append(scores)

    preds = np.array(preds)
    labels = np.array(labels)

    acc = np.mean(preds == labels)

    print(f"{PROMPT_STYLE} | {MODEL6_LORA_NAME} | {score_mode} | accuracy = {acc:.4f}")

    print("Prediction distribution:")
    display(pd.Series(preds).value_counts(normalize=True).sort_index())

    model.train()

    return {
        "prompt_style": PROMPT_STYLE,
        "lora_name": MODEL6_LORA_NAME,
        "score_mode": score_mode,
        "accuracy": acc,
        "preds": preds,
        "labels": labels,
        "scores": all_scores,
    }

Before running full training for several epochs, run a prompt ablation check to figure out which lecture length is best.

In [25]:
prompt_styles_to_test = [
    "no_lecture",
    "lecture_100",
    "lecture_180",
    "lecture_250",
]

prompt_ablation_results = []

for style in prompt_styles_to_test:
    PROMPT_STYLE = style

    print("\n" + "=" * 60)
    print("Evaluating prompt style:", PROMPT_STYLE)
    print("=" * 60)

    result = evaluate_model(
        val_df,
        max_rows=None,
        score_mode="sum",
    )

    prompt_ablation_results.append({
        "prompt_style": style,
        "accuracy": result["accuracy"],
    })

prompt_ablation_df = pd.DataFrame(prompt_ablation_results).sort_values(
    "accuracy",
    ascending=False,
)

display(prompt_ablation_df)

BEST_PROMPT_STYLE = prompt_ablation_df.iloc[0]["prompt_style"]
BEST_PROMPT_ACC = float(prompt_ablation_df.iloc[0]["accuracy"])

PROMPT_STYLE = BEST_PROMPT_STYLE

print("Best prompt style:", BEST_PROMPT_STYLE)
print("Best prompt accuracy:", BEST_PROMPT_ACC)


Evaluating prompt style: no_lecture


evaluate:   0%|          | 0/1048 [00:00<?, ?it/s]

no_lecture | attn_r8 | sum | accuracy = 0.5897
Prediction distribution:


0    0.365458
1    0.287214
2    0.304389
3    0.042939
Name: proportion, dtype: float64


Evaluating prompt style: lecture_100


...

evaluate:   0%|          | 0/1048 [00:00<?, ?it/s]

lecture_250 | attn_r8 | sum | accuracy = 0.5992
Prediction distribution:


0    0.358779
1    0.314885
2    0.287214
3    0.039122
Name: proportion, dtype: float64

,prompt_style,accuracy
1,lecture_100,0.602099
3,lecture_250,0.599237
2,lecture_180,0.595420
0,no_lecture,0.589695


Best prompt style: lecture_100
Best prompt accuracy: 0.6020992366412213


Adding a little bit of the lecture helped a lot (100 characters).

---
### Full Training

In [27]:
FULL_EPOCHS = 2  # Leave only 2 epochs to avoid overfitting
LR = cfg["lr"]
GRAD_ACCUM_STEPS = 8

MODEL5_TARGET_VAL = 0.642175572519084

RUN_NAME = f"model6_{PROMPT_STYLE}_{MODEL6_LORA_NAME}"

BEST_DIR = Path(f"../runs/{RUN_NAME}_best")
LATEST_DIR = Path(f"../runs/{RUN_NAME}_latest")
HISTORY_PATH = Path(f"../runs/{RUN_NAME}_training_history.csv")

BEST_DIR.mkdir(exist_ok=True, parents=True)
LATEST_DIR.mkdir(exist_ok=True, parents=True)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

best_model6_val_acc = -1.0
history = []

print("RUN_NAME:", RUN_NAME)
print("PROMPT_STYLE:", PROMPT_STYLE)
print("MODEL6_LORA_NAME:", MODEL6_LORA_NAME)
print("LR:", LR)
print("FULL_EPOCHS:", FULL_EPOCHS)
print("GRAD_ACCUM_STEPS:", GRAD_ACCUM_STEPS)

RUN_NAME: model6_lecture_100_attn_r8
PROMPT_STYLE: lecture_100
MODEL6_LORA_NAME: attn_r8
LR: 5e-06
FULL_EPOCHS: 2
GRAD_ACCUM_STEPS: 8


In [28]:
for epoch in range(1, FULL_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    train_order = train_df.sample(
        frac=1,
        random_state=42 + epoch,
    ).reset_index(drop=True)

    losses = []
    preds = []
    labels = []

    for step, (_, row) in enumerate(
        tqdm(train_order.iterrows(), total=len(train_order), desc=f"Model 6 epoch {epoch}")
    ):
        loss, pred, scores = contrastive_loss_for_row(row)

        (loss / GRAD_ACCUM_STEPS).backward()

        losses.append(float(loss.detach().cpu()))
        preds.append(pred)
        labels.append(int(row["answer"]))

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_order):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0,
            )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()

        if (step + 1) % 50 == 0:
            recent_loss = np.mean(losses[-50:])
            recent_acc = np.mean(np.array(preds[-100:]) == np.array(labels[-100:]))
            print(
                f"epoch {epoch}, step {step + 1}, "
                f"recent_loss={recent_loss:.4f}, recent_acc={recent_acc:.4f}"
            )

    train_loss = float(np.mean(losses))
    train_acc = float(np.mean(np.array(preds) == np.array(labels)))

    print()
    print(f"Epoch {epoch} training finished")
    print(f"train_loss = {train_loss:.6f}")
    print(f"train_acc  = {train_acc:.6f}")

    val_result = evaluate_model(
        val_df,
        max_rows=None,
        score_mode="sum",
    )

    val_acc = float(val_result["accuracy"])

    saved_best = False

    if val_acc > best_model6_val_acc:
        best_model6_val_acc = val_acc
        saved_best = True

        model.save_pretrained(BEST_DIR)
        processor.save_pretrained(BEST_DIR)

        print(f"Saved new best Model 6 checkpoint to {BEST_DIR}")

    model.save_pretrained(LATEST_DIR)
    processor.save_pretrained(LATEST_DIR)

    print(f"Saved latest Model 6 checkpoint to {LATEST_DIR}")

    history.append({
        "epoch": epoch,
        "prompt_style": PROMPT_STYLE,
        "lora_name": MODEL6_LORA_NAME,
        "lr": LR,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "saved_best": saved_best,
        "beats_model5_target": val_acc > MODEL5_TARGET_VAL,
    })

    history_df = pd.DataFrame(history)
    history_df.to_csv(HISTORY_PATH, index=False)

    display(history_df)

print("Best Model 6 validation accuracy:", best_model6_val_acc)
print("Model 5 validation accuracy:", MODEL5_TARGET_VAL)

Model 6 epoch 1:   0%|          | 0/3109 [00:00<?, ?it/s]

epoch 1, step 50, recent_loss=0.9053, recent_acc=0.6000
epoch 1, step 100, recent_loss=0.8836, recent_acc=0.6300
epoch 1, step 150, recent_loss=0.7460, recent_acc=0.6400
epoch 1, step 200, recent_loss=0.6063, recent_acc=0.6800
epoch 1, step 250, recent_loss=0.7341, recent_acc=0.7100
epoch 1, step 300, recent_loss=0.8651, recent_acc=0.6800
epoch 1, step 350, recent_loss=1.0200, recent_acc=0.5900
epoch 1, step 400, recent_loss=0.8593, recent_acc=0.5600
epoch 1, step 450, recent_loss=0.8067, recent_acc=0.6300
epoch 1, step 500, recent_loss=0.6948, recent_acc=0.6600
epoch 1, step 550, recent_loss=0.7224, recent_acc=0.6700
epoch 1, step 600, recent_loss=0.8316, recent_acc=0.6700
epoch 1, step 650, recent_loss=0.9059, recent_acc=0.6500
epoch 1, step 700, recent_loss=0.6537, recent_acc=0.6600
epoch 1, step 750, recent_loss=0.8345, recent_acc=0.6800
epoch 1, step 800, recent_loss=0.8340, recent_acc=0.6400
epoch 1, step 850, recent_loss=1.1218, recent_acc=0.5800
epoch 1, step 900, recent_loss=1

evaluate:   0%|          | 0/1048 [00:00<?, ?it/s]

lecture_100 | attn_r8 | sum | accuracy = 0.6326
Prediction distribution:


0    0.346374
1    0.341603
2    0.272901
3    0.039122
Name: proportion, dtype: float64

...

lecture_100 | attn_r8 | sum | accuracy = 0.6708
Prediction distribution:


0    0.365458
1    0.345420
2    0.247137
3    0.041985
Name: proportion, dtype: float64

Saved new best Model 6 checkpoint to ../runs/model6_lecture_100_attn_r8_best
Saved latest Model 6 checkpoint to ../runs/model6_lecture_100_attn_r8_latest


,epoch,prompt_style,lora_name,lr,train_loss,train_acc,val_acc,saved_best,beats_model5_target
0,1,lecture_100,attn_r8,0.000005,0.799701,0.658089,0.632634,True,False
1,2,lecture_100,attn_r8,0.000005,0.656042,0.714056,0.670802,True,True


Best Model 6 validation accuracy: 0.6708015267175572
Model 5 validation accuracy: 0.642175572519084


Model 6 epoch 2 beat Model 5's accuracy. Load it for submission.

---
### Submission

Load best Model 6 checkpoint before final validation / submission.

In [29]:
try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(BEST_DIR)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base_model, BEST_DIR)
model.eval()

print("Loaded best Model 6 checkpoint from:")
print(BEST_DIR)

/home/devvingduo/miniforge3/envs/pixels-to-predictions/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loaded best Model 6 checkpoint from:
../runs/model6_lecture_100_attn_r8_best


Evaluate once again on validation set.

In [30]:
best_loaded_result = evaluate_model(
    val_df,
    max_rows=None,
    score_mode="sum",
)

best_loaded_val_acc = best_loaded_result["accuracy"]

print("Best loaded Model 6 validation accuracy:", best_loaded_val_acc)
print("Model 5 target validation accuracy:", MODEL5_TARGET_VAL)

evaluate:   0%|          | 0/1048 [00:00<?, ?it/s]

lecture_100 | attn_r8 | sum | accuracy = 0.6708
Prediction distribution:


0    0.365458
1    0.345420
2    0.247137
3    0.041985
Name: proportion, dtype: float64

Best loaded Model 6 validation accuracy: 0.6708015267175572
Model 5 target validation accuracy: 0.642175572519084


Generate submission if the score beats model 5's score.

In [31]:
def predict_test(score_mode="sum"):
    model.eval()

    predictions = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="predict test"):
        pred, scores = predict_row(
            row,
            build_prompt,
            score_mode=score_mode,
        )

        predictions.append({
            "id": row["id"],
            "answer": int(pred),
        })

    return pd.DataFrame(predictions)

In [32]:
def validate_submission(submission_df, test_df):
    assert list(submission_df.columns) == ["id", "answer"]
    assert len(submission_df) == len(test_df)

    assert submission_df["id"].isna().sum() == 0
    assert submission_df["answer"].isna().sum() == 0

    submission_df["answer"] = submission_df["answer"].astype(int)

    check_df = submission_df.merge(
        test_df[["id", "num_choices"]],
        on="id",
        how="left",
    )

    assert check_df["num_choices"].isna().sum() == 0

    bad = check_df[
        (check_df["answer"] < 0) |
        (check_df["answer"] >= check_df["num_choices"])
    ]

    assert len(bad) == 0

    print("Submission validation passed.")

In [ ]:
if best_loaded_val_acc > MODEL5_TARGET_VAL:
    submission = predict_test(score_mode="sum")

    submission = submission[["id", "answer"]].copy()
    submission["answer"] = submission["answer"].astype(int)

    validate_submission(submission, test_df)

    submission.to_csv("../submission.csv", index=False)

    print("Saved submission.csv")
    print("Model 6 validation accuracy:", best_loaded_val_acc)
    print("Model 5 validation accuracy:", MODEL5_TARGET_VAL)

    display(submission.head())
else:
    print("Do not submit Model 6 yet.")
    print("Model 6 validation accuracy:", best_loaded_val_acc)
    print("Model 5 validation accuracy:", MODEL5_TARGET_VAL)

predict test:   0%|          | 0/1008 [00:00<?, ?it/s]

Submission validation passed.
Saved submission.csv
Model 6 validation accuracy: 0.6708015267175572
Model 5 validation accuracy: 0.642175572519084


,id,answer
0,test_01750,1
1,test_00128,0
2,test_02891,2
3,test_02425,2
4,test_00930,2


This model received a score of 0.70422 on Kaggle.